# Week 3 – Unsupervised Learning and Clustering Analysis

**Dataset:** UCI Wine Dataset (available through scikit-learn)

This notebook reproduces the clustering analysis used in the accompanying report. The `target` column is retained only for post-hoc validation; it is **not used** when fitting the clustering models.

In [ ]:
# Install if needed:
# %pip install pandas numpy scikit-learn matplotlib seaborn scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram

pd.set_option('display.max_columns', None)


## 1. Load and inspect the dataset

In [ ]:
# Use the local CSV included in this repository.
df = pd.read_csv('../dataset/wine.csv')
print('Shape:', df.shape)
display(df.head())
print('Missing values:', int(df.isna().sum().sum()))


In [ ]:
feature_cols = [c for c in df.columns if c != 'target']
X = df[feature_cols].copy()
y = df['target'].copy()

print('Number of features:', len(feature_cols))
print('Number of samples:', len(df))


## 2. Preprocess with standardization

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)
display(X_scaled_df.describe().round(3))


### Why standardize?
The features are measured on very different scales (for example, magnesium and proline have much larger numeric ranges than hue or phenolic measurements). K-Means is distance-based, so standardization prevents large-scale variables from dominating the clustering objective.

## 3. Choose the number of clusters: elbow + internal validation metrics

In [ ]:
metrics = []
models = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    models[k] = km
    metrics.append({
        'k': k,
        'inertia': km.inertia_,
        'silhouette': silhouette_score(X_scaled, labels),
        'calinski_harabasz': calinski_harabasz_score(X_scaled, labels),
        'davies_bouldin': davies_bouldin_score(X_scaled, labels)
    })
metrics_df = pd.DataFrame(metrics)
display(metrics_df.round(4))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(metrics_df['k'], metrics_df['inertia'], marker='o')
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Within-cluster SSE (Inertia)')
ax.set_title('Elbow Method')
ax.set_xticks(metrics_df['k'])
fig.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(metrics_df['k'], metrics_df['silhouette'], marker='o', label='Silhouette')
ax.axvline(3, linestyle='--', alpha=0.7, label='Selected k = 3')
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Silhouette score')
ax.set_title('Silhouette Score by k')
ax.set_xticks(metrics_df['k'])
ax.legend()
fig.tight_layout()
plt.show()


### Selection rationale
The elbow curve shows diminishing returns after approximately **k = 3**, while the silhouette score reaches its maximum among the tested values at **k = 3 (≈ 0.285)**. The Calinski–Harabasz index is also highest at k = 3 among the tested solutions, supporting the three-cluster choice.

## 4. Fit the final K-Means model

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
clusters = kmeans.fit_predict(X_scaled)

print('Final silhouette score:', round(silhouette_score(X_scaled, clusters), 4))
print('Final inertia:', round(kmeans.inertia_, 4))
print('Cluster counts:')
print(pd.Series(clusters, name='cluster').value_counts().sort_index())


## 5. PCA visualization

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
pca_var = pca.explained_variance_ratio_
print('PC1 explained variance:', round(pca_var[0], 4))
print('PC2 explained variance:', round(pca_var[1], 4))
print('Combined explained variance:', round(pca_var.sum(), 4))

pca_df = pd.DataFrame({'PC1': X_pca[:,0], 'PC2': X_pca[:,1], 'cluster': clusters})
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='cluster', palette='tab10', s=70, ax=ax)
ax.set_title('K-Means Clusters in PCA Space')
fig.tight_layout()
plt.show()


## 6. Cluster profiles

In [ ]:
profile = df.assign(cluster=clusters).groupby('cluster')[feature_cols].mean().round(3)
display(profile)

standard_centers = pd.DataFrame(kmeans.cluster_centers_, columns=feature_cols)
standard_centers.index.name = 'cluster'
display(standard_centers.round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.8))
sns.heatmap(standard_centers, cmap='vlag', center=0, annot=True, fmt='.2f', ax=ax)
ax.set_title('Standardized K-Means Cluster Centers')
fig.tight_layout()
plt.show()


## 7. Silhouette plot for the selected solution

In [ ]:
from sklearn.metrics import silhouette_samples

sample_scores = silhouette_samples(X_scaled, clusters)
fig, ax = plt.subplots(figsize=(8, 5.5))
y_lower = 10
for i in range(3):
    vals = np.sort(sample_scores[clusters == i])
    size_i = len(vals)
    y_upper = y_lower + size_i
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals, alpha=0.7)
    ax.text(-0.05, (y_lower + y_upper) / 2, str(i))
    y_lower = y_upper + 10
ax.axvline(silhouette_score(X_scaled, clusters), linestyle='--', label='Average silhouette')
ax.set_xlabel('Silhouette coefficient')
ax.set_ylabel('Cluster')
ax.set_title('Silhouette Plot (k=3)')
ax.legend()
fig.tight_layout()
plt.show()


## 8. Hierarchical clustering cross-check

In [ ]:
Z = linkage(X_scaled, method='ward')
fig, ax = plt.subplots(figsize=(11, 5.5))
dendrogram(Z, truncate_mode='lastp', p=25, leaf_rotation=90, leaf_font_size=8, ax=ax)
ax.set_title('Hierarchical Clustering Dendrogram (Ward Linkage)')
ax.set_xlabel('Merged observations / clusters')
ax.set_ylabel('Distance')
fig.tight_layout()
plt.show()

agglomerative = AgglomerativeClustering(n_clusters=3, linkage='ward')
hier_labels = agglomerative.fit_predict(X_scaled)
print('Hierarchical silhouette:', round(silhouette_score(X_scaled, hier_labels), 4))
print('K-Means vs hierarchical ARI:', round(adjusted_rand_score(clusters, hier_labels), 4))


## 9. Post-hoc validation against the known reference labels

In [ ]:
# IMPORTANT: target labels are not used for training. This is only a validation check because the dataset provides reference classes.
ari = adjusted_rand_score(y, clusters)
print('Adjusted Rand Index (K-Means clusters vs reference labels):', round(ari, 4))


## 10. Export the key results

In [ ]:
results_dir = '../results'

metrics_df.to_csv(f'{results_dir}/k_selection_metrics_reproduced.csv', index=False)
pd.DataFrame({'Cluster': np.arange(3), 'Count': np.bincount(clusters)}).to_csv(f'{results_dir}/cluster_sizes_reproduced.csv', index=False)
profile.reset_index().rename(columns={'cluster':'Cluster'}).to_csv(f'{results_dir}/cluster_profiles_reproduced.csv', index=False)
standard_centers.reset_index().rename(columns={'cluster':'Cluster'}).to_csv(f'{results_dir}/cluster_centers_standardized_reproduced.csv', index=False)

assignments = df.copy()
assignments['cluster'] = clusters
assignments.to_csv(f'{results_dir}/wine_cluster_assignments_reproduced.csv', index=False)
print('Results exported to', results_dir)


## Conclusion

The standardized Wine Dataset is best represented by **three clusters** under the tested K-Means solutions. The clustering is reasonably separated rather than perfectly separated, as reflected by the silhouette score of about **0.285**. PCA and hierarchical clustering provide complementary views of the same structure, while the reference-label ARI can be used only as a post-hoc validation metric because the problem is unsupervised.